In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("TP1-Decouverte-Spark")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())

sc = spark.sparkContext
print("Version Spark :", spark.version)
print("Master        :", sc.master)
print("Parallélisme  :", sc.defaultParallelism)

Version Spark : 4.2.0
Master        : local[4]
Parallélisme  : 4


In [2]:
myRange = spark.range(1000).toDF("number")
myRange.show(5)
myRange.printSchema()
print("Nombre total de lignes :", myRange.count())

+------+
|number|
+------+
|     0|
|     1|
|     2|
|     3|
|     4|
+------+
only showing top 5 rows
root
 |-- number: long (nullable = false)

Nombre total de lignes : 1000


In [3]:
data = [("Alice", 29), ("Bob", 31), ("Catherine", 35)]
df = spark.createDataFrame(data, ["name", "age"])

df.show()
df.printSchema()

+---------+---+
|     name|age|
+---------+---+
|    Alice| 29|
|      Bob| 31|
|Catherine| 35|
+---------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)



In [4]:
from pyspark.sql.functions import col

data += [("David", 41), ("Emma", 27), ("Frank", 38)]
df = spark.createDataFrame(data, ["name", "age"])

df.show()
df.printSchema()
print("Total :", df.count())
df.select("name").show()
df.filter(col("age") > 30).show()

+---------+---+
|     name|age|
+---------+---+
|    Alice| 29|
|      Bob| 31|
|Catherine| 35|
|    David| 41|
|     Emma| 27|
|    Frank| 38|
+---------+---+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)

Total : 6
+---------+
|     name|
+---------+
|    Alice|
|      Bob|
|Catherine|
|    David|
|     Emma|
|    Frank|
+---------+

+---------+---+
|     name|age|
+---------+---+
|      Bob| 31|
|Catherine| 35|
|    David| 41|
|    Frank| 38|
+---------+---+



In [5]:
print("Nombre de partitions :", df.rdd.getNumPartitions())
print("Contenu réparti :", df.rdd.glom().collect())

Nombre de partitions : 4
Contenu réparti : [[Row(name='Alice', age=29)], [Row(name='Bob', age=31), Row(name='Catherine', age=35)], [Row(name='David', age=41)], [Row(name='Emma', age=27), Row(name='Frank', age=38)]]


In [6]:
df2 = df.repartition(2)
print("Nouvelles partitions :", df2.rdd.getNumPartitions())

Nouvelles partitions : 2


In [7]:
from pyspark.sql import functions as F

big = spark.range(0, 2_000_000, 1, 8).withColumn("bucket", F.col("id") % 5)

# Transformations uniquement
step1 = big.filter(F.col("id") % 2 == 0)
step2 = step1.select("id", "bucket")
step3 = step2.groupBy("bucket").count()

print("Transformations déclarées, mais aucun calcul n'a encore eu lieu !")

Transformations déclarées, mais aucun calcul n'a encore eu lieu !


In [8]:
step3.show()

+------+------+
|bucket| count|
+------+------+
|     2|200000|
|     4|200000|
|     0|200000|
|     1|200000|
|     3|200000|
+------+------+

